# 02B — Block Encoder Training and Full-Corpus Prediction

Pipeline stage:
- load labeled block sample
- train a single block-level AI classifier
- validate and select an operating threshold
- apply the model to the full block corpus
- export predicted AI-positive blocks for the next stage

Output directory

`output/02B_block_train/`
- `model/`
- `block_predictions/part_*.parquet`
- `ai_blocks.parquet`

In [1]:
!pip -q install pyarrow datasets transformers accelerate scikit-learn tqdm

## Environment Setup

In [2]:
import os
import gc
import math
import inspect
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.dataset as ds
from tqdm.auto import tqdm

import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

import datasets
from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    set_seed,
)

from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
BASE_DIR = "/content/drive/MyDrive/NLP_FINAL_PROJECT_Tom_Chen"
IN_DIR_02A = f"{BASE_DIR}/output/02A_block_sample"
OUT_DIR_02B = f"{BASE_DIR}/output/02B_block_train"

LABELED_BLOCKS_PARQUET = f"{IN_DIR_02A}/block_labels.parquet"
BLOCKS_DIR = f"{IN_DIR_02A}/blocks"

MODEL_DIR = f"{OUT_DIR_02B}/model"
PRED_DIR = f"{OUT_DIR_02B}/block_predictions"
AI_BLOCKS_PARQUET = f"{OUT_DIR_02B}/ai_blocks.parquet"

os.makedirs(OUT_DIR_02B, exist_ok=True)
os.makedirs(PRED_DIR, exist_ok=True)

assert os.path.exists(LABELED_BLOCKS_PARQUET), f"Missing: {LABELED_BLOCKS_PARQUET}"
assert os.path.exists(BLOCKS_DIR), f"Missing: {BLOCKS_DIR}"

## Configuration

In [4]:
SEED = 42
set_seed(SEED)
np.random.seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
GPU_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NA"
USE_BF16 = bool(torch.cuda.is_available())
USE_FP16 = False

if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

print("device:", DEVICE)
print("gpu:", GPU_NAME)
print("torch:", torch.__version__)
print("transformers:", __import__("transformers").__version__)
print("USE_BF16:", USE_BF16)

device: cuda
gpu: NVIDIA A100-SXM4-80GB
torch: 2.10.0+cu128
transformers: 5.0.0
USE_BF16: True


In [5]:
BACKBONE = "distilroberta-base"

MAX_LENGTH = 256
NUM_EPOCHS = 3
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01

PER_DEVICE_TRAIN_BATCH = 32
PER_DEVICE_EVAL_BATCH = 64
GRAD_ACCUM = 1

INFER_BATCH_SIZE = 256
INFER_TABLE_BATCH_ROWS = 50_000

VALID_SIZE = 0.15

THRESHOLD_GRID = np.arange(0.20, 0.91, 0.02)

LENGTH_BUCKETS = [
    (0, 80),
    (81, 160),
    (161, 256),
    (257, 384),
    (385, 10_000_000),
]

## Load Labeled Block Sample

In [6]:
lab = pd.read_parquet(LABELED_BLOCKS_PARQUET)
print("raw labeled shape:", lab.shape)

required_cols = [
    "doc_id",
    "block_id",
    "block_uid",
    "block_text",
    "is_ai_block",
    "label_status",
]
missing = [c for c in required_cols if c not in lab.columns]
assert not missing, f"Missing required columns: {missing}"

lab = lab[lab["label_status"] == "ok"].copy()
lab["text"] = lab["block_text"].fillna("").astype(str)
lab["label"] = pd.to_numeric(lab["is_ai_block"], errors="coerce").fillna(0).astype(int)

print("usable labeled rows:", len(lab))
print(lab["label"].value_counts(dropna=False))

raw labeled shape: (12000, 16)
usable labeled rows: 12000
label
0    7778
1    4222
Name: count, dtype: int64


## Train / Validation Split

In [7]:
train_df, valid_df = train_test_split(
    lab[["text", "label"]].copy(),
    test_size=VALID_SIZE,
    random_state=SEED,
    stratify=lab["label"],
)

train_df = train_df.reset_index(drop=True)
valid_df = valid_df.reset_index(drop=True)

print("train shape:", train_df.shape)
print("valid shape:", valid_df.shape)
print("\ntrain label distribution:")
print(train_df["label"].value_counts(dropna=False))
print("\nvalid label distribution:")
print(valid_df["label"].value_counts(dropna=False))

train shape: (10200, 2)
valid shape: (1800, 2)

train label distribution:
label
0    6611
1    3589
Name: count, dtype: int64

valid label distribution:
label
0    1167
1     633
Name: count, dtype: int64


## Utility Functions

In [8]:
def make_hf_dataset(df: pd.DataFrame) -> Dataset:
    return Dataset.from_pandas(df[["text", "label"]], preserve_index=False)

def tokenize_batch(examples: Dict[str, List[str]], tokenizer):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=MAX_LENGTH,
    )

def softmax_pos_prob(logits: np.ndarray) -> np.ndarray:
    x = logits - logits.max(axis=1, keepdims=True)
    ex = np.exp(x)
    p = ex / ex.sum(axis=1, keepdims=True)
    return p[:, 1]

def compute_binary_metrics(y_true: np.ndarray, p_pos: np.ndarray, threshold: float) -> Dict[str, float]:
    y_pred = (p_pos >= threshold).astype(int)
    p, r, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="binary", zero_division=0
    )
    acc = accuracy_score(y_true, y_pred)
    return {
        "threshold": float(threshold),
        "accuracy": float(acc),
        "precision": float(p),
        "recall": float(r),
        "f1": float(f1),
    }

def search_best_threshold(y_true: np.ndarray, p_pos: np.ndarray, grid: np.ndarray) -> Dict[str, float]:
    best = None
    for t in grid:
        m = compute_binary_metrics(y_true, p_pos, float(t))
        if best is None or m["f1"] > best["f1"]:
            best = m
    return best

def compute_metrics_default(eval_pred):
    logits, labels = eval_pred
    probs = softmax_pos_prob(np.asarray(logits))
    y_pred = (probs >= 0.5).astype(int)
    p, r, f1, _ = precision_recall_fscore_support(
        labels, y_pred, average="binary", zero_division=0
    )
    acc = accuracy_score(labels, y_pred)
    return {
        "accuracy": float(acc),
        "precision": float(p),
        "recall": float(r),
        "f1": float(f1),
    }

def build_training_args(output_dir: str) -> TrainingArguments:
    sig = inspect.signature(TrainingArguments.__init__)
    valid = set(sig.parameters.keys())

    kwargs = {
        "output_dir": output_dir,
        "num_train_epochs": NUM_EPOCHS,
        "learning_rate": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,
        "per_device_train_batch_size": PER_DEVICE_TRAIN_BATCH,
        "per_device_eval_batch_size": PER_DEVICE_EVAL_BATCH,
        "gradient_accumulation_steps": GRAD_ACCUM,
        "logging_strategy": "steps",
        "logging_steps": 50,
        "save_strategy": "no",
        "report_to": "none",
        "fp16": USE_FP16,
        "bf16": USE_BF16,
        "remove_unused_columns": True,
    }

    if "eval_strategy" in valid:
        kwargs["eval_strategy"] = "epoch"
    elif "evaluation_strategy" in valid:
        kwargs["evaluation_strategy"] = "epoch"

    kwargs = {k: v for k, v in kwargs.items() if k in valid}
    return TrainingArguments(**kwargs)

def sanity_forward_pass(model, tokenizer, texts: List[str], labels: List[int]) -> None:
    model = model.to(DEVICE).eval()
    enc = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt",
    )
    enc = {k: v.to(DEVICE) for k, v in enc.items()}
    enc["labels"] = torch.tensor(labels, dtype=torch.long, device=DEVICE)

    with torch.no_grad():
        out = model(**enc)
        print("logits finite:", torch.isfinite(out.logits).all().item())
        print("loss finite:", torch.isfinite(out.loss).item(), "| loss:", float(out.loss.detach().cpu().item()))

def infer_probs(model, tokenizer, texts: List[str], batch_size: int) -> np.ndarray:
    out = []
    with torch.inference_mode():
        for i in range(0, len(texts), batch_size):
            batch_texts = texts[i:i + batch_size]
            enc = tokenizer(
                batch_texts,
                padding=True,
                truncation=True,
                max_length=MAX_LENGTH,
                return_tensors="pt",
            )
            enc = {k: v.to(DEVICE) for k, v in enc.items()}
            logits = model(**enc).logits
            probs = (
                torch.softmax(logits, dim=1)[:, 1]
                .detach()
                .cpu()
                .numpy()
                .astype(np.float32)
            )
            out.append(probs)
    return np.concatenate(out) if out else np.array([], dtype=np.float32)

def bucket_indices_by_length(lengths: np.ndarray, buckets: List[Tuple[int, int]]) -> List[np.ndarray]:
    idx_groups = []
    for lo, hi in buckets:
        idx = np.where((lengths >= lo) & (lengths <= hi))[0]
        if len(idx) > 0:
            idx_groups.append(idx)
    return idx_groups

## Tokenizer and Model

In [9]:
tokenizer = AutoTokenizer.from_pretrained(BACKBONE)
model = AutoModelForSequenceClassification.from_pretrained(BACKBONE, num_labels=2)

sanity_forward_pass(
    model,
    tokenizer,
    texts=train_df["text"].head(4).tolist(),
    labels=train_df["label"].head(4).tolist(),
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/480 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/331M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/101 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: distilroberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
classifier.dense.bias       | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.out_proj.weight  | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


logits finite: True
loss finite: True | loss: 0.634795606136322


## Dataset Preparation

In [10]:
ds_train = make_hf_dataset(train_df).map(lambda x: tokenize_batch(x, tokenizer), batched=True)
ds_valid = make_hf_dataset(valid_df).map(lambda x: tokenize_batch(x, tokenizer), batched=True)

ds_train = ds_train.remove_columns(["text"]).rename_column("label", "labels")
ds_valid = ds_valid.remove_columns(["text"]).rename_column("label", "labels")

ds_train = ds_train.cast_column("labels", datasets.Value("int64"))
ds_valid = ds_valid.cast_column("labels", datasets.Value("int64"))

collator = DataCollatorWithPadding(tokenizer=tokenizer)

Map:   0%|          | 0/10200 [00:00<?, ? examples/s]

Map:   0%|          | 0/1800 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/10200 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/1800 [00:00<?, ? examples/s]

## Training

In [11]:
training_args = build_training_args(output_dir=OUT_DIR_02B)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=ds_train,
    eval_dataset=ds_valid,
    data_collator=collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics_default,
)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.236418,0.238444,0.907778,0.903282,0.826224,0.863036
2,0.195330,0.246024,0.901667,0.875000,0.840442,0.857373
3,0.150312,0.268829,0.903889,0.883333,0.837283,0.859692


TrainOutput(global_step=957, training_loss=0.21839008124519915, metrics={'train_runtime': 38.9521, 'train_samples_per_second': 785.581, 'train_steps_per_second': 24.569, 'total_flos': 2009646346585344.0, 'train_loss': 0.21839008124519915, 'epoch': 3.0})

## Validation and Threshold Selection

In [12]:
pred_valid = trainer.predict(ds_valid)
p_valid = softmax_pos_prob(np.asarray(pred_valid.predictions))
y_valid = np.asarray(pred_valid.label_ids)

best_threshold = search_best_threshold(y_valid, p_valid, THRESHOLD_GRID)
print("best threshold:")
print(best_threshold)

valid_pred_label = (p_valid >= best_threshold["threshold"]).astype(int)

valid_predictions = valid_df.copy()
valid_predictions["p_ai_block"] = p_valid
valid_predictions["pred_ai_block"] = valid_pred_label

print("\nvalidation prediction distribution:")
print(valid_predictions["pred_ai_block"].value_counts(dropna=False))

display(valid_predictions.head(15))

best threshold:
{'threshold': 0.5799999999999998, 'accuracy': 0.9066666666666666, 'precision': 0.8960817717206133, 'recall': 0.8309636650868878, 'f1': 0.8622950819672132}

validation prediction distribution:
pred_ai_block
0    1213
1     587
Name: count, dtype: int64


,text,label,p_ai_block,pred_ai_block
0,Sheppard Mullin Richter & Hampton,0,0.001897,0
1,Space Magnetic Structures Observed Near Superm...,0,0.004134,0
2,Renowned technologists and entrepreneurs Neha ...,1,0.996972,1
3,Hacking Communications Mathematical Modeling,0,0.133415,0
4,Skewed data entrenches rifts with Muslims and ...,1,0.045016,0
5,Exploring the Impact of AI on Nematode Quarant...,0,0.991938,1
6,Entrepreneurship Greek Debt Crisis See all cha...,0,0.002473,0
7,Technology MagazinesWhy subscribe?The best tec...,0,0.030909,0
8,Data Mining Image Recognition Signal Recognition,1,0.750918,1
9,Auala Fou: Fausia e Saienitisi Saina Robotic A...,1,0.886027,1


## Save Final Model

In [13]:
trainer.save_model(MODEL_DIR)
tokenizer.save_pretrained(MODEL_DIR)

print("wrote:", MODEL_DIR)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

wrote: /content/drive/MyDrive/NLP_FINAL_PROJECT_Tom_Chen/output/02B_block_train/model


## Full-Corpus Prediction

In [14]:
tokenizer_inf = AutoTokenizer.from_pretrained(MODEL_DIR)
model_inf = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR).to(DEVICE).eval()

P_AI = float(best_threshold["threshold"])
print("inference threshold:", P_AI)

BLOCK_COLS = [
    "doc_id",
    "block_id",
    "block_uid",
    "url",
    "date",
    "language",
    "title",
    "domain",
    "time_bucket",
    "block_text",
    "block_len",
    "ai_kw_flag",
]

block_files = sorted(
    [
        os.path.join(BLOCKS_DIR, f)
        for f in os.listdir(BLOCKS_DIR)
        if f.endswith(".parquet")
    ]
)

print("block shard count:", len(block_files))

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

inference threshold: 0.5799999999999998
block shard count: 23


In [15]:
for shard_path in tqdm(block_files, desc="Shards", position=0):
    shard_name = os.path.basename(shard_path).replace(".parquet", "")
    pf = pq.ParquetFile(shard_path)
    n_parts = math.ceil(pf.metadata.num_rows / INFER_TABLE_BATCH_ROWS)

    for part_j, batch in tqdm(
        enumerate(
            pf.iter_batches(
                batch_size=INFER_TABLE_BATCH_ROWS,
                columns=BLOCK_COLS,
            )
        ),
        total=n_parts,
        desc=shard_name,
        position=1,
        leave=False,
    ):
        out_path = os.path.join(PRED_DIR, f"{shard_name}_part{part_j:03d}_pred.parquet")

        if os.path.exists(out_path):
            continue

        pdf = batch.to_pandas()
        pdf["block_text"] = pdf["block_text"].fillna("").astype(str)

        if "block_len" not in pdf.columns:
            pdf["block_len"] = pdf["block_text"].str.len().astype(int)

        texts = pdf["block_text"].tolist()
        lengths = pdf["block_len"].to_numpy()

        p_ai = np.zeros(len(pdf), dtype=np.float32)
        pred_ai = np.zeros(len(pdf), dtype=np.int8)

        idx_groups = bucket_indices_by_length(lengths, LENGTH_BUCKETS)

        for idx in idx_groups:
            sub_texts = [texts[k] for k in idx]
            p_sub = infer_probs(
                model_inf,
                tokenizer_inf,
                sub_texts,
                batch_size=INFER_BATCH_SIZE,
            )
            p_ai[idx] = p_sub
            pred_ai[idx] = (p_sub >= P_AI).astype(np.int8)

        pdf["p_ai_block"] = p_ai
        pdf["pred_ai_block"] = pred_ai

        pdf.to_parquet(out_path, index=False)

        del pdf
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

print("prediction shards written:", PRED_DIR)

Shards:   0%|          | 0/23 [00:00<?, ?it/s]

part_0000:   0%|          | 0/5 [00:00<?, ?it/s]

part_0001:   0%|          | 0/5 [00:00<?, ?it/s]

part_0002:   0%|          | 0/5 [00:00<?, ?it/s]

part_0003:   0%|          | 0/5 [00:00<?, ?it/s]

...

part_0019:   0%|          | 0/5 [00:00<?, ?it/s]

part_0020:   0%|          | 0/5 [00:00<?, ?it/s]

part_0021:   0%|          | 0/5 [00:00<?, ?it/s]

part_0022:   0%|          | 0/1 [00:00<?, ?it/s]

prediction shards written: /content/drive/MyDrive/NLP_FINAL_PROJECT_Tom_Chen/output/02B_block_train/block_predictions


## Build AI-Positive Block Table

In [16]:
pred_files = sorted(
    [
        os.path.join(PRED_DIR, f)
        for f in os.listdir(PRED_DIR)
        if f.endswith(".parquet")
    ]
)

assert pred_files, f"No prediction shards found in: {PRED_DIR}"

writer = None
rows_written = 0

for fpath in tqdm(pred_files, desc="Collecting AI-positive blocks"):
    part = pd.read_parquet(fpath)
    keep = part[part["pred_ai_block"] == 1].copy()

    if len(keep) == 0:
        continue

    table = pa.Table.from_pandas(keep, preserve_index=False)

    if writer is None:
        writer = pq.ParquetWriter(
            AI_BLOCKS_PARQUET,
            table.schema,
            compression="zstd",
        )

    writer.write_table(table)
    rows_written += len(keep)

    del part, keep, table
    gc.collect()

if writer is not None:
    writer.close()

print("wrote:", AI_BLOCKS_PARQUET)
print("ai-positive rows:", rows_written)

wrote: /content/drive/MyDrive/NLP_FINAL_PROJECT_Tom_Chen/output/02B_block_train/ai_blocks.parquet
ai-positive rows: 1184289


## Quick Readout

In [17]:
ai_blocks_preview = pd.read_parquet(AI_BLOCKS_PARQUET)
print("ai_blocks shape:", ai_blocks_preview.shape)

print("\nAI block prediction distribution in final retained set:")
print(ai_blocks_preview["pred_ai_block"].value_counts(dropna=False))

display(
    ai_blocks_preview[
        [
            "doc_id",
            "block_id",
            "domain",
            "title",
            "block_text",
            "p_ai_block",
        ]
    ].head(15)
)

ai_blocks shape: (1184289, 14)

AI block prediction distribution in final retained set:
pred_ai_block
1    1184289
Name: count, dtype: int64


,doc_id,block_id,domain,title,block_text,p_ai_block
0,0,0,blockworks.co,"Bad Idea AI Price (BAD), Market Cap, Price Tod...","Bad Idea AI Price (BAD), Market Cap, Price Tod...",0.990286
1,1,0,boingboing.net,This AI video of gymnastics might be the freak...,This AI video of gymnastics might be the freak...,0.996618
2,1,5,boingboing.net,This AI video of gymnastics might be the freak...,Thumbnails : Youtube Thumbnail generator This ...,0.997094
3,1,6,boingboing.net,This AI video of gymnastics might be the freak...,"Jul 1, 2024 Screenshot: Luma AI",0.994060
4,1,7,boingboing.net,This AI video of gymnastics might be the freak...,I'm sure by now you're tired of looking at ter...,0.998320
5,1,8,boingboing.net,This AI video of gymnastics might be the freak...,Imagine showing these to a tribe that's never ...,0.998448
6,1,11,boingboing.net,This AI video of gymnastics might be the freak...,CW: Body Horror? This AI video attempt to show...,0.998307
7,1,12,boingboing.net,This AI video of gymnastics might be the freak...,AI ai videos freaky grostesque gymnastics Stev...,0.995146
8,2,0,boingboing.net,"If using AI feels like a chore, try this - Boi...","If using AI feels like a chore, try this - Boi...",0.996764
9,2,5,boingboing.net,"If using AI feels like a chore, try this - Boi...",Thumbnails : Youtube Thumbnail generator If us...,0.997387
